# Demonstracja ETL: Dane przed i po
Poniższy notatnik pozwala przetestować na żywo, jak dane wyglądały przed wyczyszczeniem (w strefie Staging) oraz jak wyglądają w hurtowni (Data Warehouse) po transformacji.

In [ ]:
import os
import sys
# Dodajemy ścieżkę projektu do sys.path, aby móc importować moduły z src
sys.path.append(os.path.abspath('..'))

import pandas as pd
from src.utils.db import sqlserver_connection

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Zależności załadowane pomyślnie!")

## Funkcja pomocnicza do zapytań
Przygotujmy prostą funkcję, która pobierze dane z SQL Server do obiektu DataFrame.

In [ ]:
def query_db(sql_query: str) -> pd.DataFrame:
    with sqlserver_connection() as conn:
        return pd.read_sql(sql_query, conn)

## Krok 1: Dane Surowe (Staging) - "Przed"
Sprawdźmy, jak wyglądają surowe dane przed zastosowaniem procesów ETL.

In [ ]:
staging_sql = """
SELECT TOP 5 
    invoice_and_item_number,
    date,
    store_location,      -- Format przestrzenny POINT
    category_name,       -- Zwróć uwagę na braki danych (NULL)
    state_bottle_cost,   -- Format z symbolem waluty $
    sale_dollars         -- Format z symbolem waluty $
FROM stg.iowa_liquor_sales_raw;
"""
df_stg = query_db(staging_sql)
df_stg.head()

## Krok 2: Oczyszczone wymiary i fakty - "Po"
Zobaczmy jak dane zostały przetransformowane do czystej i spójnej postaci w schemacie `dw` (Data Warehouse).

In [ ]:
store_sql = """
SELECT TOP 5 
    store_key, 
    store_name, 
    latitude, 
    longitude 
FROM dw.dim_store;
"""
df_store = query_db(store_sql)
print("Wyodrębnione współrzędne w wymiarze sklepu:")
display(df_store)

In [ ]:
fact_sql = """
SELECT TOP 5 
    invoice_number,
    sale_dollars,       -- Teraz to czysty DECIMAL
    state_bottle_cost,  -- Teraz to czysty DECIMAL
    bottles_sold,
    margin_amount       -- Wyliczona nowa miara pochodna
FROM dw.fact_sales;
"""
df_fact = query_db(fact_sql)
print("Oczyszczone finanse i wyliczona marża w tabeli faktów:")
display(df_fact)

### Obsługa braków danych
Zobaczmy, czy brakujące przypisania kategorii (które w stg były puste lub NULL) zostały obsłużone za pomocą rekordu `UNKNOWN`.

In [ ]:
category_sql = """
SELECT * 
FROM dw.dim_category
WHERE category_number = 'UNKNOWN';
"""
df_category = query_db(category_sql)
display(df_category)

## Krok 3: Warstwa semantyczna - Gotowe raporty
Ostatecznie nasza aplikacja odpytuje tylko bezpieczne widoki semantyczne (schemat `sem`), które hermetyzują logikę i dają czyste wyniki dla biznesu.

In [ ]:
semantic_sql = """
SELECT TOP 10 * 
FROM sem.vw_sales_by_month
ORDER BY year, month;
"""
df_sem = query_db(semantic_sql)
display(df_sem)